In [115]:
#cria objeto "MapaTabelas que para cada município cria uma lista ordenada dos dados do município"
#sendo igual as posições a tabela será a mesma para diferentes municípios (na teoria)
from pathlib import Path

MapaTabelas = {}

def chave(p):
    partes = p.stem.split("-")
    return (
        int(partes[partes.index("page") + 1]),
        int(partes[partes.index("table") + 1]),
    )

for municipio_folder in sorted(Path("municipios").iterdir()):
    arquivos = sorted(municipio_folder.glob("*.csv"), key=chave)
    MapaTabelas[municipio_folder.name] = arquivos


print(MapaTabelas)

{'Abaetetuba': [PosixPath('municipios/Abaetetuba/Abaetetuba-page-15-table-1.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-15-table-2.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-15-table-3.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-16-table-1.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-17-table-1.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-18-table-1.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-18-table-2.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-18-table-3.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-19-table-1.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-19-table-2.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-19-table-3.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-19-table-4.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-20-table-1.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-page-20-table-2.csv'), PosixPath('municipios/Abaetetuba/Abaetetuba-pa

In [116]:
from pathlib import Path
import shutil

#cria pasta tabelas de cada município sobre respectivo índice 
def cria_tabelas_pop(pos,path):
    Path('pop_area')
    Path('dataIn').mkdir(exist_ok=True)
    caminho = Path('dataIn') / path
    caminho.mkdir(exist_ok=True)

    for municipio in MapaTabelas.keys():
        origem = MapaTabelas[municipio][pos]
        destino = caminho / f"{municipio}.csv"
        shutil.copy2(origem, destino)

    

In [117]:
path = 'pop_test'
cria_tabelas_pop(1,path)

In [ ]:
import pandas as pd
import os
def procura_erros(path):
    erros = []
    dataFrames = []
    for municipio in sorted(os.listdir(path)):
        df = pd.read_csv(f'{path}/{municipio}')

        if municipio == 'Abaetetuba.csv':
            dfPadrao = pd.read_csv(f'{path}/{municipio}')
            colunasPadrao = dfPadrao.columns.to_list()

        colunas = df.columns.to_list()
        
        if any(col in colunas for col in colunasPadrao):
                dataFrames.append(df)
        else:
            erros.append(municipio)

    return erros

In [123]:
procura_erros('dataIn/pop_area')

[]

In [128]:
import os
import pandas as pd

def junta_tudo(path):
    lst = []
    for table in os.listdir(path):
        name, ext = table.split('.')
        caminho = f'{path}/{table}'
        df = pd.read_csv(caminho)
        df['municipio'] = name
        lst.append(df)
    
    df_junto = pd.concat(lst,ignore_index=True)

    return df_junto

In [ ]:
def cria_tabelao(pos, path):
    

In [129]:
a = junta_tudo('dataIn/pop_area')

a

,Anos,População (Hab.),Área (Km²),Densidade (Hab./Km²),municipio,Área (km²),Densidade (Hab./km²),População (hab.),Densidade (hab./km²)
0,2000,17.193,"6.368,30","2,71",Brasil-Novo,NaN,NaN,NaN,NaN
1,2001(1),17.93,"6.368,30","2,82",Brasil-Novo,NaN,NaN,NaN,NaN
2,2002(1),18.387,"6.368,30",2.89,Brasil-Novo,NaN,NaN,NaN,NaN
3,2003(1),18.941,"6.368,30","2,97",Brasil-Novo,NaN,NaN,NaN,NaN
4,2004(1),20.197,"6.368,30","3,17",Brasil-Novo,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
3581,2020(1),20.525,"3.444,29","5,96",Floresta-do-Araguaia,NaN,NaN,NaN,NaN
3582,2021(1),20.742,"3.444,29","6,02",Floresta-do-Araguaia,NaN,NaN,NaN,NaN
3583,2022,17.898,"3.444,29","5,20",Floresta-do-Araguaia,NaN,NaN,NaN,NaN
3584,2023(2),18.625,"3.444,29","5,41",Floresta-do-Araguaia,NaN,NaN,NaN,NaN


In [120]:
import os
import pandas as pd
path = Path('dataIn/pop_area')


lst = []
for table in os.listdir(path):
    name, ext = table.split('.')
    df = pd.read_csv(path / table)
    df['municipio'] = name
    lst.append(df)

df_junto = pd.concat(lst,ignore_index=True)

df = df_junto.copy()

df.columns = [
    'Ano', 'Pop', 'Area', 'Densidade',
    'municipio', 'area_2', 'densidade_2',
    'populacao_2', 'densidade_3'
]

df['Ano'] = df['Ano'].astype(str).str[:4]

df['Pop'] = df['Pop'].combine_first(df['populacao_2'])
df['Area'] = df['Area'].combine_first(df['area_2'])
df['Densidade'] = (
    df['Densidade']
    .combine_first(df['densidade_2'])
    .combine_first(df['densidade_3'])
)

df = df.drop(columns=['populacao_2', 'area_2', 'densidade_2','densidade_3'])

Path('dadosOut').mkdir(exist_ok=True)
df.to_csv('dadosOut/area.csv',index=False)